# RNA annotation 

This notebook imports the count tables (output from TEtranscripts) and assigns each gene to a class of RNA.

In [2]:
version

               _                           
platform       aarch64-apple-darwin20      
arch           aarch64                     
os             darwin20                    
system         aarch64, darwin20           
status                                     
major          4                           
minor          5.0                         
year           2025                        
month          04                          
day            11                          
svn rev        88135                       
language       R                           
version.string R version 4.5.0 (2025-04-11)
nickname       How About a Twenty-Six      

In [ ]:
setwd("..")

In [3]:
renv::init()
renv::activate()
renv::status()

- The project is out-of-sync -- use `renv::status()` for details.
The following package(s) are in an inconsistent state:

 package          installed recorded used
 AnnotationDbi    y         n        y   
 askpass          y         n        y   
 Biobase          y         n        y   
 BiocFileCache    y         n        y   
 BiocGenerics     y         n        y   
 BiocManager      y         n        y   
 BiocVersion      y         n        y   
 biomaRt          y         n        y   
 Biostrings       y         n        y   
 bit              y         n        y   
 bit64            y         n        y   
 blob             y         n        y   
 cachem           y         n        y   
 cli              y         n        y   
 crayon           y         n        y   
 curl             y         n        y   
 data.table       y         n        y   
 DBI              y         n        y   
 dbplyr           y         n        y   
 digest           y         n        y

# Imports

In [4]:
library(dplyr)
library(tidyr)
library(stringr)
library(data.table)
library(ggplot2)
library(biomaRt)
library(AnnotationDbi)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last


Loading required package: stats4

Loading required package: BiocGenerics

Loading required package: generics


Attaching package: ‘generics’


The following object is masked from ‘package:dplyr’:

    explain


The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: ‘BiocGenerics’


The following object is masked from ‘package:dplyr’:

    combine


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    coln

In [ ]:
setwd("TECounts")

In [5]:
# Create list of all .cntTable files in TEcounts_output folder
sample_files <- list.files("TEcounts_output", pattern = "*.cntTable", full.names = TRUE)

# Filter out l1-5 (low quality)
sample_files <- sample_files[!grepl("l1-5", basename(sample_files))]

# Annotate genes

In [6]:
get_gene_annotations <- function(gene_ids) {
  tryCatch({
    # Connect to Ensembl mouse database
    mart <- useMart("ensembl", dataset = "mmusculus_gene_ensembl")
    
    # Clean gene IDs (remove version numbers if present)
    clean_gene_ids <- gsub("\\.[0-9]+$", "", gene_ids)
    
    # Get gene biotypes
    annotations <- getBM(
      attributes = c("ensembl_gene_id", "gene_biotype", "external_gene_name"),
      filters = "ensembl_gene_id",
      values = clean_gene_ids,
      mart = mart
    )
    
    # Check if successful
    if(nrow(annotations) == 0) {
      warning("No annotations found.")
      annotations <- data.frame(
        ensembl_gene_id = clean_gene_ids,
        gene_biotype = "unknown",
        external_gene_name = clean_gene_ids,
        stringsAsFactors = FALSE
      )
    }
    
    return(annotations)
  }, error = function(e) {
    warning("biomaRt failed")
  })
}

# Analysis across all samples
get_overall_rna_types_annotated <- function(sample_files) {
  # Combine all samples
  all_samples_combined <- do.call(rbind, lapply(sample_files, function(cnt_file) {
    cnt_data <- fread(cnt_file, header = TRUE)
    colnames(cnt_data) <- c("gene_id", "counts")
    cnt_data$gene_id <- gsub('"', '', cnt_data$gene_id)
    return(cnt_data)
  }))
  
  # Get unique gene IDs
  unique_gene_ids <- unique(all_samples_combined$gene_id)
  
  # Get annotations
  annotations <- get_gene_annotations(unique_gene_ids)
  
  # Clean gene IDs in count data for joining
  all_samples_combined$gene_id_clean <- gsub("\\.[0-9]+$", "", all_samples_combined$gene_id)
  
  # Merge with count data
  all_samples_annotated <- all_samples_combined %>%
    left_join(annotations, by = c("gene_id_clean" = "ensembl_gene_id"))
  
  # Calculate total reads
  total_reads_all <- sum(all_samples_annotated$counts)
  
  # Group by biotype
  overall_rna_types <- all_samples_annotated %>%
    group_by(gene_biotype) %>%
    summarise(
      total_counts = sum(counts),
      percentage = (total_counts / total_reads_all) * 100,
      rpm = (total_counts / total_reads_all) * 1000000,
      .groups = 'drop'
    ) %>%
    arrange(desc(percentage))
  
  return(overall_rna_types)
}

# Run the analysis
overall_rna_types <- get_overall_rna_types_annotated(sample_files)

# Save results
write.csv(overall_rna_types, "TEresults/overall_rna_types_annotated.csv", row.names = FALSE)

### Plot summary of RNA types

In [7]:
p1 <- ggplot(overall_rna_types, aes(x = reorder(gene_biotype, -percentage), y = percentage, fill = gene_biotype)) +
  geom_bar(stat = "identity") +
  geom_text(aes(label = sprintf("%.2f%%", percentage)), 
            vjust = -0.5, size = 3) +
  labs(title = "RNA Types Distribution (Properly Annotated)", 
       x = "Gene Biotype", y = "Percentage (%)") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1),
        legend.position = "none")

# Save plot
ggsave("TEresults/overall_rna_types_annotated.png", p1, width = 12, height = 8)

print("Overall RNA Types Summary (Properly Annotated):")
print(overall_rna_types)

[1] "Overall RNA Types Summary (Properly Annotated):"
# A tibble: 37 × 4
   gene_biotype                       total_counts percentage     rpm
   <chr>                                     <int>      <dbl>   <dbl>
 1 protein_coding                        703656755    57.4    573658.
 2 NA                                    478486809    39.0    390088.
 3 lncRNA                                 25568836     2.08    20845.
 4 TEC                                     7934135     0.647    6468.
 5 unprocessed_pseudogene                  3863635     0.315    3150.
 6 processed_pseudogene                    2864828     0.234    2336.
 7 ribozyme                                1102399     0.0899    899.
 8 transcribed_unprocessed_pseudogene       748470     0.0610    610.
 9 transcribed_processed_pseudogene         556183     0.0453    453.
10 miRNA                                    541697     0.0442    442.
# ℹ 27 more rows


# Analyze retrovirus levels

In [8]:
# Function to analyze retrovirus in more detail
analyze_retrovirus <- function(cnt_file) {
  cnt_data <- fread(cnt_file, header = TRUE)
  colnames(cnt_data) <- c("gene_id", "counts")
  cnt_data$gene_id <- gsub('"', '', cnt_data$gene_id)
  
  # Extract sample name from filename
  sample_name <- gsub(".cntTable", "", basename(cnt_file))
  
  # Calculate total reads for normalization
  total_reads <- sum(cnt_data$counts)
  
  # Identify retroviral elements (LTR elements, especially ERVs)
  retroviral_elements <- cnt_data[grepl(":LTR:", cnt_data$gene_id) | 
                                 grepl("ERV", cnt_data$gene_id), ]
  
  # Calculate retrovirus metrics
  retrovirus_counts <- sum(retroviral_elements$counts)
  retrovirus_percent <- (retrovirus_counts / total_reads) * 100
  retrovirus_rpm <- (retrovirus_counts / total_reads) * 1000000
  
  # Breakdown by ERV family
  erv_families <- retroviral_elements %>%
    mutate(erv_family = case_when(
      grepl("ERVK", gene_id) ~ "ERVK",
      grepl("ERVL", gene_id) ~ "ERVL", 
      grepl("ERV1", gene_id) ~ "ERV1",
      grepl("ERV2", gene_id) ~ "ERV2",
      grepl("ERV3", gene_id) ~ "ERV3",
      TRUE ~ "Other_LTR"
    )) %>%
    group_by(erv_family) %>%
    summarise(counts = sum(counts), .groups = 'drop') %>%
    mutate(rpm = (counts / total_reads) * 1000000)
  
  return(list(
    sample = sample_name,
    total_reads = total_reads,
    retrovirus_counts = retrovirus_counts,
    retrovirus_percent = retrovirus_percent,
    retrovirus_rpm = retrovirus_rpm,
    erv_families = erv_families
  ))
}

# Analyze each sample
sample_results <- lapply(sample_files, analyze_retrovirus)

# Create comparison table
comparison_table <- do.call(rbind, lapply(sample_results, function(x) {
  data.frame(
    Sample = x$sample,
    Total_Reads = x$total_reads,
    Retrovirus_Counts = x$retrovirus_counts,
    Retrovirus_Percent = x$retrovirus_percent,
    Retrovirus_RPM = x$retrovirus_rpm
  )
}))

# Extract condition and replicate from sample names
comparison_table <- comparison_table %>%
  mutate(
    condition = gsub("-.*", "", Sample),
    replicate = gsub(".*-", "", Sample)
  )

# Create ERV family comparison
erv_comparison <- do.call(rbind, lapply(sample_results, function(x) {
  x$erv_families$sample <- x$sample
  x$erv_families
}))

# Add condition and replicate to ERV comparison
erv_comparison <- erv_comparison %>%
  mutate(
    condition = gsub("-.*", "", sample),
    replicate = gsub(".*-", "", sample)
  )

# Save results
write.csv(comparison_table, "TEresults/retrovirus_comparison.csv", row.names = FALSE)
write.csv(erv_comparison, "TEresults/erv_family_comparison.csv", row.names = FALSE)

### Plot retrovirus composition

In [ ]:
p2 <- ggplot(comparison_table, aes(x = Sample, y = Retrovirus_Percent, fill = condition)) +
  geom_bar(stat = "identity") +
  geom_text(aes(label = sprintf("%.2f%%", Retrovirus_Percent)), 
            vjust = -0.5, size = 2.5) +
  labs(title = "Retrovirus Percentage by Sample", 
       x = "Sample", y = "Retrovirus Percentage (%)") +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))



In [10]:
ggsave("TEresults/retrovirus_per_sample.png", p2, width = 12, height = 8)

In [ ]:
# ERV family breakdown 
p3 <- ggplot(erv_comparison, aes(x = condition, y = rpm, fill = erv_family)) +
  geom_bar(stat = "identity", position = "stack") +
  labs(title = "ERV Family Breakdown by Condition", 
       x = "Condition", y = "RPM", fill = "ERV Family") +
  theme_minimal() +
  scale_fill_brewer(palette = "Set1")

In [14]:
ggsave("TEresults/retrovirus_breakdown.png", p3, width = 12, height = 8)

In [17]:
p4 <- ggplot(comparison_table, aes(x = condition, y = Retrovirus_RPM, fill = condition)) +
  geom_bar(stat = "identity", position = "dodge", alpha = 0.3) +
  geom_point(aes(group = replicate), position = position_dodge(width = 0.9), 
             size = 3, color = "black") +
  labs(title = "Retrovirus RPM by Condition", 
       x = "Condition", y = "Retrovirus RPM") +
  theme_minimal() +
  theme(legend.position = "none")

In [18]:
ggsave("TEresults/retrovirus_count.png", p4, width = 12, height = 8)